# HalfCourt - Logistic Regression Training

Train the Logistic Regression baseline on the HalfCourt halftime dataset and save `logistic_regression.joblib`.

**Note:** Logistic Regression does not have a margin prediction head. When used via the CLI, the report will indicate that margin of victory is unavailable for this model.

## 1. Install dependencies

In [ ]:
!pip install -q shap scikit-learn pandas pyarrow joblib

## 2. Upload dataset

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## 3. Set configuration

In [ ]:
PARQUET_PATH = 'halfcourt_training.parquet'
MODEL_OUT = 'logistic_regression.joblib'
TEST_SIZE = 0.2
RANDOM_STATE = 42

MAX_ITER = 1000
C = 1.0

## 4. Load dataset

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_parquet(PARQUET_PATH)
df['game_date'] = pd.to_datetime(df['game_date'])
df = df.sort_values('game_date').reset_index(drop=True)

print(f'Dataset shape : {df.shape}')
print(f'Date range    : {df["game_date"].min().date()} to {df["game_date"].max().date()}')
print(f'Home win rate : {df["home_win"].mean():.3f}')

## 5. Temporal train / test split + scaling

Logistic Regression is sensitive to feature scale, so we apply `StandardScaler` fitted on train only.

In [ ]:
from sklearn.preprocessing import StandardScaler

FEATURE_COLS = [
    'ht_score_diff', 'q1_diff', 'q2_diff', 'momentum_shift',
    'home_fg_pct', 'home_3p_pct', 'home_ft_pct', 'home_efg_pct', 'home_ts_pct',
    'away_fg_pct', 'away_3p_pct', 'away_ft_pct', 'away_efg_pct', 'away_ts_pct',
    'home_to', 'away_to', 'home_ast_to_ratio', 'away_ast_to_ratio',
    'home_oreb_pct', 'away_oreb_pct', 'reb_diff',
    'home_pace_est', 'away_pace_est',
    'home_bench_pts', 'away_bench_pts', 'home_bench_ratio', 'away_bench_ratio',
    'home_starters_3foul', 'away_starters_3foul', 'foul_diff',
    'home_star1_pts', 'home_star1_ast', 'home_star_foul_flag',
    'away_star1_pts', 'away_star1_ast', 'away_star_foul_flag',
    'is_home', 'home_rest_days', 'away_rest_days',
    'home_back2back', 'away_back2back', 'is_playoffs',
]
LABEL_COL = 'home_win'

split_idx = int(len(df) * (1 - TEST_SIZE))
train_df  = df.iloc[:split_idx]
test_df   = df.iloc[split_idx:]

X_train_raw = train_df[FEATURE_COLS].values.astype(np.float32)
X_test_raw  = test_df[FEATURE_COLS].values.astype(np.float32)
y_train     = train_df[LABEL_COL].values.astype(np.float32)
y_test      = test_df[LABEL_COL].values.astype(np.float32)

scaler  = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_test  = scaler.transform(X_test_raw)

print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')

## 6. Train Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(
    max_iter=MAX_ITER,
    C=C,
    random_state=RANDOM_STATE,
)
model.fit(X_train, y_train)
print('Training complete.')

## 7. Evaluation report

In [ ]:
import shap
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, brier_score_loss

proba = model.predict_proba(X_test)[:, 1]
preds = (proba >= 0.5).astype(int)

print('=== HalfCourt Model Evaluation ===')
print(f'Model type     : logistic_regression')
print(f'Train samples  : {len(X_train)}')
print(f'Test samples   : {len(X_test)}')
print()
print(f'Accuracy       : {accuracy_score(y_test, preds):.3f}')
print(f'AUC-ROC        : {roc_auc_score(y_test, proba):.3f}')
print(f'F1 Score       : {f1_score(y_test, preds):.3f}')
print(f'Brier Score    : {brier_score_loss(y_test, proba):.3f}')
print()

explainer   = shap.LinearExplainer(model, X_train)
shap_values = explainer.shap_values(X_test)
mean_abs    = np.abs(shap_values).mean(axis=0)
ranked      = sorted(zip(FEATURE_COLS, mean_abs), key=lambda x: x[1], reverse=True)[:10]

print('Top 10 Features by SHAP importance:')
for rank, (feat, val) in enumerate(ranked, 1):
    print(f'  {rank:2d}. {feat:<30} {val:.3f}')

## 8. Coefficient plot

In [ ]:
import matplotlib.pyplot as plt

coefs = model.coef_[0]
sorted_idx = np.argsort(np.abs(coefs))[::-1][:20]
sorted_feats = [FEATURE_COLS[i] for i in sorted_idx]
sorted_coefs = coefs[sorted_idx]

colors = ['steelblue' if c > 0 else 'tomato' for c in sorted_coefs]
plt.figure(figsize=(10, 6))
plt.barh(sorted_feats[::-1], sorted_coefs[::-1], color=colors[::-1])
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Coefficient (positive = favors home win)')
plt.title('Logistic Regression: Top 20 Coefficients')
plt.tight_layout()
plt.savefig('logistic_regression_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to logistic_regression_coefficients.png')

## 9. SHAP summary plot

In [ ]:
shap.summary_plot(shap_values, X_test, feature_names=FEATURE_COLS, show=False)
plt.tight_layout()
plt.savefig('logistic_regression_shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to logistic_regression_shap_summary.png')

## 10. Bundle scaler + model and save

The scaler fitted on the training set must be saved alongside the model so that inference in `predictor.py` applies the same transformation.

In [ ]:
import joblib
import sys
sys.path.insert(0, '/content')  # ensure halfcourt package is findable if uploaded

# Import from the canonical module so the class pickles with a stable path
# that predictor.py can resolve when the model is loaded locally.
# If halfcourt is not on the Colab path, fall back to defining it inline
# under the same qualified name so the remap patch in predictor.py still works.
try:
    from halfcourt.model_trainer import ScaledLogisticRegression
except ModuleNotFoundError:
    import halfcourt  # noqa: F401 — ensure package stub exists
    class ScaledLogisticRegression:
        def __init__(self, scaler, model):
            self.scaler = scaler
            self.model  = model
        def predict_proba(self, X):
            return self.model.predict_proba(self.scaler.transform(X))
    # Register under the canonical module path so joblib serialises correctly
    import types
    _mod = sys.modules.get("halfcourt.model_trainer")
    if _mod is None:
        _mod = types.ModuleType("halfcourt.model_trainer")
        sys.modules["halfcourt.model_trainer"] = _mod
    _mod.ScaledLogisticRegression = ScaledLogisticRegression
    ScaledLogisticRegression.__module__ = "halfcourt.model_trainer"
    ScaledLogisticRegression.__qualname__ = "ScaledLogisticRegression"

wrapped = ScaledLogisticRegression(scaler, model)
joblib.dump(wrapped, MODEL_OUT)
print(f"Model saved -> {MODEL_OUT}")

# Quick sanity check
test_proba = wrapped.predict_proba(X_test_raw)[:, 1]
from sklearn.metrics import roc_auc_score
print(f"Sanity AUC (wrapped) : {roc_auc_score(y_test, test_proba):.3f}")


## 11. Download

Download `logistic_regression.joblib` and place it in `models/saved/` in your local repo.

In [ ]:
files.download(MODEL_OUT)
files.download('logistic_regression_coefficients.png')
files.download('logistic_regression_shap_summary.png')